# ROGII P2-P02（本地按井五折 OOF 10.3057）提交 Notebook

本 Notebook 只运行已冻结的 P2-P02：12 个基础特征 + 24 个物理候选特征 + 5 条 128-seed PF 路径特征。五个 fold 均为同一组 LightGBM 参数，推理时取预测均值。不会读取训练集中的同名井、隐藏 TVT、formation surface 或同井覆盖信息。

In [ ]:
from pathlib import Path
import glob
import time
import numpy as np
import pandas as pd
from numba import njit
import lightgbm as lgb

# P2-P02 固定的 41 列顺序。五个 Booster 内部只保存 Column_0..40，顺序不能改变。
FEATURE_COLUMNS = ['last_visible_tvt', 'md_since_visible_end', 'hidden_fraction', 'x_current', 'y_current', 'z_current', 'dx_from_visible_end', 'dy_from_visible_end', 'dz_from_visible_end', 'dxy_from_visible_end', 'gr_raw', 'gr_missing', 'pf_ancc_delta', 'pf_ancc_std', 'pf_z_delta', 'pf_vs_z', 'beam_cons_d', 'beam_loose_d', 'beam_vcons_d', 'beam_sm5_d', 'beam_vloose_d', 'beam_mid_d', 'beam_stiff_d', 'beam_mean_d', 'beam_std_d', 'beam_med_d', 'sc8_d', 'sc8_sc', 'sc15_d', 'sc15_sc', 'sc25_d', 'sc25_sc', 'sc_cons_d', 'sc_ens_d', 'sc_trust', 'hyb_d', 'pf128_mean_delta', 'pf128_scale_3_delta', 'pf128_scale_5_delta', 'pf128_scale_8_delta', 'pf128_scale_12_delta']

DIRECT_CANDIDATE_COLUMNS = ['pf_ancc_delta', 'pf_ancc_std', 'pf_z_delta', 'pf_vs_z', 'beam_cons_d', 'beam_loose_d', 'beam_vcons_d', 'beam_sm5_d', 'beam_vloose_d', 'beam_mid_d', 'beam_stiff_d', 'beam_mean_d', 'beam_std_d', 'beam_med_d', 'sc8_d', 'sc8_sc', 'sc15_d', 'sc15_sc', 'sc25_d', 'sc25_sc', 'sc_cons_d', 'sc_ens_d', 'sc_trust', 'hyb_d']

# 128-seed PF 的冻结参数，来自 P2-P01/P02 正式配置。
PF_PARAMETERS = {'number_of_particles': 500, 'number_of_seeds': 128, 'seed_base': 0, 'typewell_grid_step_ft': 0.2, 'initial_position_spread_ft': 4.5, 'initial_rate_std': 0.01, 'rate_momentum': 0.998, 'rate_noise': 0.002, 'position_noise_ft': 0.005, 'position_limit_beyond_typewell_ft': 100.0, 'minimum_md_step_ft': 1.0, 'squared_gr_residual_cap': 600.0, 'likelihood_floor': 1e-300, 'gr_sigma_min_api': 10.0, 'gr_sigma_max_api': 60.0, 'initial_rate_visible_tail_rows': 30, 'resample_effective_fraction': 0.5, 'resample_position_noise_ft': 0.1, 'resample_rate_noise': 0.001, 'likelihood_scales': [3.0, 5.0, 8.0, 12.0], 'formal_seed_aggregation': 'unweighted_mean'}

print("固定方案：P2-P02，41 特征，5 个同配置 LightGBM fold 模型取均值")
print("本地按井五折 OOF micro RMSE：10.3057049921")


## 1. C01 的 24 个确定性物理候选特征

In [ ]:
"""从原始单井数据复现 notebook 的 24 个 PF、Beam 与 NCC 候选特征。"""

from __future__ import annotations

import numpy as np
import pandas as pd
from numba import njit



PF_N = 600
ANCC_N = 600
PF_MOM = 0.993
PF_VN = 0.005
PF_PN = 0.01
PF_GR_SIG_MIN = 10.0
PF_GR_SIG_MAX = 60.0
PF_GR_SIG_DEFAULT = 30.0
PF_RESAMPLE_THRESHOLD = 0.5
PF_ROUGH_POSITION = 0.2
PF_ROUGH_VELOCITY = 0.003
PF_GR_WINDOW = 5
PF_SMOOTH_GR_WEIGHT = 0.3
ANCC_ALPHA = 0.998
ANCC_RATE_NOISE = 0.002
ANCC_POSITION_NOISE = 0.005
ANCC_INITIAL_SPREAD = 0.3
ANCC_ROUGH_POSITION = 0.1
ANCC_ROUGH_RATE = 0.001

BEAM_CONFIGS = (
    (10, 20.0, 144.0, 2, "cons"),
    (10, 8.0, 64.0, 2, "loose"),
    (8, 35.0, 220.0, 1, "vcons"),
    (10, 14.0, 90.0, 5, "sm5"),
    (20, 4.0, 36.0, 3, "vloose"),
    (12, 12.0, 100.0, 3, "mid"),
    (15, 25.0, 180.0, 2, "stiff"),
)


@njit(cache=False)
def _interp_regular_grid(
    grid: np.ndarray,
    value: float,
    minimum: float,
    step: float,
) -> float:
    """在等间隔 Typewell 网格上做线性插值。"""

    left_index = int((value - minimum) / step)
    if left_index < 0:
        return grid[0]
    final_index = len(grid) - 1
    if left_index >= final_index:
        return grid[final_index]
    fraction = (value - minimum) / step - left_index
    return grid[left_index] * (1.0 - fraction) + grid[left_index + 1] * fraction


@njit(cache=False)
def _systematic_resample(
    positions: np.ndarray,
    auxiliary_state: np.ndarray,
    weights: np.ndarray,
    particle_count: int,
    position_roughness: float,
    auxiliary_roughness: float,
) -> tuple[np.ndarray, np.ndarray]:
    """按权重系统重采样，并给复制出的状态加入 notebook 原始粗化噪声。"""

    cumulative = np.zeros(particle_count + 1)
    for particle_index in range(particle_count):
        cumulative[particle_index + 1] = (
            cumulative[particle_index] + weights[particle_index]
        )
    first_sample = np.random.uniform(0.0, 1.0 / particle_count)
    new_positions = np.empty(particle_count)
    new_auxiliary = np.empty(particle_count)
    source_index = 0
    for particle_index in range(particle_count):
        sample_position = first_sample + particle_index / particle_count
        while (
            source_index < particle_count - 1
            and cumulative[source_index + 1] < sample_position
        ):
            source_index += 1
        new_positions[particle_index] = (
            positions[source_index] + position_roughness * np.random.randn()
        )
        new_auxiliary[particle_index] = (
            auxiliary_state[source_index] + auxiliary_roughness * np.random.randn()
        )
    return new_positions, new_auxiliary


@njit(cache=False)
def _particle_filter_ancc(
    hidden_md: np.ndarray,
    hidden_z: np.ndarray,
    hidden_gr: np.ndarray,
    typewell_gr_grid: np.ndarray,
    grid_minimum: float,
    grid_step: float,
    gr_sigma: float,
    initial_structure_position: float,
    initial_structure_rate: float,
    particle_count: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """追踪 U=TVT+Z；显式 seed 是相对原 notebook 的可复现性修正。"""

    np.random.seed(seed)
    positions = np.empty(particle_count)
    rates = np.empty(particle_count)
    weights = np.ones(particle_count) / particle_count
    for particle_index in range(particle_count):
        positions[particle_index] = (
            initial_structure_position + ANCC_INITIAL_SPREAD * np.random.randn()
        )
        rates[particle_index] = initial_structure_rate + 0.01 * np.random.randn()

    predictions = np.empty(len(hidden_md))
    standard_deviations = np.empty(len(hidden_md))
    previous_md = hidden_md[0] - 1.0
    maximum_tvt = grid_minimum + len(typewell_gr_grid) * grid_step + 50.0
    minimum_tvt = grid_minimum - 50.0

    for row_index in range(len(hidden_md)):
        md_step = max(hidden_md[row_index] - previous_md, 1.0)
        for particle_index in range(particle_count):
            rates[particle_index] = (
                ANCC_ALPHA * rates[particle_index]
                + ANCC_RATE_NOISE * np.random.randn()
            )
            positions[particle_index] += (
                rates[particle_index] * md_step
                + ANCC_POSITION_NOISE * np.random.randn()
            )
            particle_tvt = positions[particle_index] - hidden_z[row_index]
            particle_tvt = max(particle_tvt, minimum_tvt)
            particle_tvt = min(particle_tvt, maximum_tvt)
            positions[particle_index] = particle_tvt + hidden_z[row_index]

        if not np.isnan(hidden_gr[row_index]):
            weight_sum = 0.0
            for particle_index in range(particle_count):
                expected_gr = _interp_regular_grid(
                    typewell_gr_grid,
                    positions[particle_index] - hidden_z[row_index],
                    grid_minimum,
                    grid_step,
                )
                normalized_error = (hidden_gr[row_index] - expected_gr) / gr_sigma
                squared_error = normalized_error * normalized_error
                likelihood = (
                    np.exp(-0.5 * squared_error) if squared_error < 600.0 else 0.0
                )
                likelihood = max(likelihood, 1e-300)
                weights[particle_index] *= likelihood
                weight_sum += weights[particle_index]
            if weight_sum > 0.0:
                for particle_index in range(particle_count):
                    weights[particle_index] /= weight_sum
            else:
                for particle_index in range(particle_count):
                    weights[particle_index] = 1.0 / particle_count

        squared_weight_sum = 0.0
        for particle_index in range(particle_count):
            squared_weight_sum += weights[particle_index] * weights[particle_index]
        if 1.0 / squared_weight_sum < PF_RESAMPLE_THRESHOLD * particle_count:
            positions, rates = _systematic_resample(
                positions,
                rates,
                weights,
                particle_count,
                ANCC_ROUGH_POSITION,
                ANCC_ROUGH_RATE,
            )
            for particle_index in range(particle_count):
                weights[particle_index] = 1.0 / particle_count

        predicted_tvt = 0.0
        for particle_index in range(particle_count):
            predicted_tvt += weights[particle_index] * (
                positions[particle_index] - hidden_z[row_index]
            )
        predictions[row_index] = predicted_tvt
        variance = 0.0
        for particle_index in range(particle_count):
            difference = (
                positions[particle_index] - hidden_z[row_index] - predicted_tvt
            )
            variance += weights[particle_index] * difference * difference
        standard_deviations[row_index] = variance**0.5
        previous_md = hidden_md[row_index]

    return predictions, standard_deviations


@njit(cache=False)
def _particle_filter_z(
    hidden_md: np.ndarray,
    hidden_z: np.ndarray,
    hidden_gr: np.ndarray,
    hidden_smoothed_gr: np.ndarray,
    typewell_gr_grid: np.ndarray,
    typewell_smoothed_gr_grid: np.ndarray,
    grid_minimum: float,
    grid_step: float,
    gr_sigma: float,
    initial_tvt: float,
    initial_velocity: float,
    z_velocity_coefficient: float,
    z_velocity_intercept: float,
    z_velocity_sigma: float,
    particle_count: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """追踪 TVT 和 dTVT/dMD；显式 seed 保证同值重跑一致。"""

    np.random.seed(seed)
    positions = np.empty(particle_count)
    velocities = np.empty(particle_count)
    weights = np.ones(particle_count) / particle_count
    for particle_index in range(particle_count):
        positions[particle_index] = initial_tvt + 0.5 * np.random.randn()
        velocities[particle_index] = initial_velocity + 0.02 * np.random.randn()

    predictions = np.empty(len(hidden_md))
    standard_deviations = np.empty(len(hidden_md))
    previous_md = hidden_md[0] - 1.0
    previous_z = hidden_z[0] - 1.0
    maximum_tvt = grid_minimum + len(typewell_gr_grid) * grid_step + 50.0
    minimum_tvt = grid_minimum - 50.0

    for row_index in range(len(hidden_md)):
        md_step = max(hidden_md[row_index] - previous_md, 1.0)
        z_change_per_md = (hidden_z[row_index] - previous_z) / md_step
        expected_velocity = (
            z_velocity_coefficient * z_change_per_md + z_velocity_intercept
        )
        for particle_index in range(particle_count):
            velocities[particle_index] = (
                PF_MOM * velocities[particle_index] + PF_VN * np.random.randn()
            )
            positions[particle_index] += (
                velocities[particle_index] * md_step + PF_PN * np.random.randn()
            )
            positions[particle_index] = max(positions[particle_index], minimum_tvt)
            positions[particle_index] = min(positions[particle_index], maximum_tvt)

        if not np.isnan(hidden_gr[row_index]):
            weight_sum = 0.0
            for particle_index in range(particle_count):
                expected_raw_gr = _interp_regular_grid(
                    typewell_gr_grid,
                    positions[particle_index],
                    grid_minimum,
                    grid_step,
                )
                raw_error = (hidden_gr[row_index] - expected_raw_gr) / gr_sigma
                raw_squared_error = raw_error * raw_error
                raw_likelihood = (
                    np.exp(-0.5 * raw_squared_error)
                    if raw_squared_error < 600.0
                    else 0.0
                )
                raw_likelihood = max(raw_likelihood, 1e-300)
                if not np.isnan(hidden_smoothed_gr[row_index]):
                    expected_smoothed_gr = _interp_regular_grid(
                        typewell_smoothed_gr_grid,
                        positions[particle_index],
                        grid_minimum,
                        grid_step,
                    )
                    smooth_error = (
                        hidden_smoothed_gr[row_index] - expected_smoothed_gr
                    ) / (gr_sigma * 1.5)
                    smooth_squared_error = smooth_error * smooth_error
                    smooth_likelihood = (
                        np.exp(-0.5 * smooth_squared_error)
                        if smooth_squared_error < 600.0
                        else 0.0
                    )
                    smooth_likelihood = max(smooth_likelihood, 1e-300)
                    likelihood = (
                        (1.0 - PF_SMOOTH_GR_WEIGHT) * raw_likelihood
                        + PF_SMOOTH_GR_WEIGHT * smooth_likelihood
                    )
                else:
                    likelihood = raw_likelihood
                weights[particle_index] *= max(likelihood, 1e-300)
                weight_sum += weights[particle_index]
            if weight_sum > 0.0:
                for particle_index in range(particle_count):
                    weights[particle_index] /= weight_sum
            else:
                for particle_index in range(particle_count):
                    weights[particle_index] = 1.0 / particle_count

        velocity_weight_sum = 0.0
        velocity_scale = max(z_velocity_sigma * 2.0, 0.005)
        for particle_index in range(particle_count):
            velocity_error = (
                velocities[particle_index] - expected_velocity
            ) / velocity_scale
            velocity_squared_error = velocity_error * velocity_error
            velocity_likelihood = (
                np.exp(-0.5 * velocity_squared_error)
                if velocity_squared_error < 600.0
                else 0.0
            )
            velocity_likelihood = max(velocity_likelihood, 1e-300)
            weights[particle_index] *= velocity_likelihood
            velocity_weight_sum += weights[particle_index]
        if velocity_weight_sum > 0.0:
            for particle_index in range(particle_count):
                weights[particle_index] /= velocity_weight_sum
        else:
            for particle_index in range(particle_count):
                weights[particle_index] = 1.0 / particle_count

        squared_weight_sum = 0.0
        for particle_index in range(particle_count):
            squared_weight_sum += weights[particle_index] * weights[particle_index]
        if 1.0 / squared_weight_sum < PF_RESAMPLE_THRESHOLD * particle_count:
            positions, velocities = _systematic_resample(
                positions,
                velocities,
                weights,
                particle_count,
                PF_ROUGH_POSITION,
                PF_ROUGH_VELOCITY,
            )
            for particle_index in range(particle_count):
                weights[particle_index] = 1.0 / particle_count

        predicted_tvt = 0.0
        for particle_index in range(particle_count):
            predicted_tvt += weights[particle_index] * positions[particle_index]
        predictions[row_index] = predicted_tvt
        variance = 0.0
        for particle_index in range(particle_count):
            difference = positions[particle_index] - predicted_tvt
            variance += weights[particle_index] * difference * difference
        standard_deviations[row_index] = variance**0.5
        previous_md = hidden_md[row_index]
        previous_z = hidden_z[row_index]

    return predictions, standard_deviations


@njit(cache=False)
def _beam_search_indices(
    smoothed_gr: np.ndarray,
    typewell_gr: np.ndarray,
    start_index: int,
    beam_size: int,
    movement_cost: float,
    error_scale: float,
) -> np.ndarray:
    """复现 notebook 的允许 -2 到 +2 移动并最终回溯的 Beam Search。"""

    row_count = len(smoothed_gr)
    typewell_count = len(typewell_gr)
    maximum_candidates = beam_size * 6
    beam_indices = np.zeros(beam_size, np.int64)
    beam_indices[0] = start_index
    beam_costs = np.full(beam_size, 1e30)
    beam_costs[0] = 0.0
    active_beams = np.int64(1)
    history_indices = np.zeros((row_count, beam_size), np.int64)
    history_parents = np.zeros((row_count, beam_size), np.int64)
    candidate_indices = np.zeros(maximum_candidates, np.int64)
    candidate_costs = np.full(maximum_candidates, 1e30)
    candidate_parents = np.zeros(maximum_candidates, np.int64)

    for row_index in range(row_count):
        observed_gr = smoothed_gr[row_index]
        candidate_count = np.int64(0)
        for beam_index in range(active_beams):
            current_index = beam_indices[beam_index]
            current_cost = beam_costs[beam_index]
            for movement in range(-2, 3):
                next_index = current_index + movement
                if next_index < 0 or next_index >= typewell_count:
                    continue
                absolute_movement = movement if movement >= 0 else -movement
                total_cost = (
                    current_cost
                    + (observed_gr - typewell_gr[next_index]) ** 2 / error_scale
                    + movement_cost * absolute_movement
                )
                found_index = np.int64(-1)
                for candidate_index in range(candidate_count):
                    if candidate_indices[candidate_index] == next_index:
                        found_index = candidate_index
                        break
                if found_index >= 0:
                    if total_cost < candidate_costs[found_index]:
                        candidate_costs[found_index] = total_cost
                        candidate_parents[found_index] = beam_index
                elif candidate_count < maximum_candidates:
                    candidate_indices[candidate_count] = next_index
                    candidate_costs[candidate_count] = total_cost
                    candidate_parents[candidate_count] = beam_index
                    candidate_count += 1

        kept_count = min(beam_size, candidate_count)
        for kept_index in range(kept_count):
            minimum_index = kept_index
            for candidate_index in range(kept_index + 1, candidate_count):
                if candidate_costs[candidate_index] < candidate_costs[minimum_index]:
                    minimum_index = candidate_index
            if minimum_index != kept_index:
                candidate_indices[kept_index], candidate_indices[minimum_index] = (
                    candidate_indices[minimum_index],
                    candidate_indices[kept_index],
                )
                candidate_costs[kept_index], candidate_costs[minimum_index] = (
                    candidate_costs[minimum_index],
                    candidate_costs[kept_index],
                )
                candidate_parents[kept_index], candidate_parents[minimum_index] = (
                    candidate_parents[minimum_index],
                    candidate_parents[kept_index],
                )
        history_indices[row_index, :kept_count] = candidate_indices[:kept_count]
        history_parents[row_index, :kept_count] = candidate_parents[:kept_count]
        beam_indices[:kept_count] = candidate_indices[:kept_count]
        beam_costs[:kept_count] = candidate_costs[:kept_count]
        active_beams = kept_count

    best_beam = np.int64(0)
    for beam_index in range(1, active_beams):
        if beam_costs[beam_index] < beam_costs[best_beam]:
            best_beam = beam_index
    path = np.zeros(row_count, np.int64)
    current_beam = best_beam
    for row_index in range(row_count - 1, -1, -1):
        path[row_index] = history_indices[row_index, current_beam]
        current_beam = history_parents[row_index, current_beam]
    return path


def _regular_typewell_grid(
    typewell_tvt: np.ndarray,
    typewell_gr: np.ndarray,
    step: float = 0.2,
) -> tuple[np.ndarray, float, float]:
    minimum = float(typewell_tvt.min())
    maximum = float(typewell_tvt.max())
    tvt_grid = np.arange(minimum, maximum + step, step)
    gr_grid = np.interp(tvt_grid, typewell_tvt, typewell_gr).astype(np.float64)
    return gr_grid, minimum, float(step)


def _gr_sigma(
    horizontal_df: pd.DataFrame,
    typewell_tvt: np.ndarray,
    typewell_gr: np.ndarray,
) -> float:
    visible = horizontal_df[
        horizontal_df["TVT_input"].notna() & horizontal_df["GR"].notna()
    ]
    if len(visible) < 20:
        return PF_GR_SIG_DEFAULT
    expected_gr = np.interp(
        visible["TVT_input"].to_numpy(dtype=np.float64),
        typewell_tvt,
        typewell_gr,
    )
    residual = visible["GR"].to_numpy(dtype=np.float64) - expected_gr
    return float(np.clip(np.std(residual), PF_GR_SIG_MIN, PF_GR_SIG_MAX))


def _nearest_index(sorted_values: np.ndarray, value: float) -> int:
    insertion_index = int(np.searchsorted(sorted_values, value, side="left"))
    if insertion_index >= len(sorted_values):
        return len(sorted_values) - 1
    if insertion_index > 0:
        left_distance = abs(sorted_values[insertion_index - 1] - value)
        right_distance = abs(sorted_values[insertion_index] - value)
        if left_distance <= right_distance:
            return insertion_index - 1
    return insertion_index


def _smooth_gr(values: np.ndarray, fallback: float, radius: int) -> np.ndarray:
    series = pd.Series(values, dtype="float32")
    series = series.interpolate(limit_direction="both").fillna(fallback)
    if radius > 0:
        series = series.rolling(
            radius * 2 + 1,
            center=True,
            min_periods=1,
        ).mean()
    return series.to_numpy(dtype=np.float32)


def _beam_search(
    horizontal_gr: np.ndarray,
    typewell_tvt: np.ndarray,
    typewell_gr: np.ndarray,
    start_tvt: float,
    beam_size: int,
    movement_cost: float,
    error_scale: float,
    smoothing_radius: int,
) -> np.ndarray:
    start_index = _nearest_index(typewell_tvt, start_tvt)
    smoothed_gr = _smooth_gr(
        horizontal_gr,
        float(np.nanmean(typewell_gr)),
        smoothing_radius,
    ).astype(np.float64)
    path_indices = _beam_search_indices(
        smoothed_gr,
        typewell_gr.astype(np.float64),
        start_index,
        beam_size,
        float(movement_cost),
        float(error_scale),
    )
    return typewell_tvt[path_indices].astype(np.float32)


def _multi_scale_ncc(
    visible_gr: np.ndarray,
    visible_tvt: np.ndarray,
    hidden_gr: np.ndarray,
    half_windows: tuple[int, ...] = (8, 15, 25),
    stride: int = 3,
) -> tuple[list[tuple[np.ndarray, np.ndarray]], np.ndarray]:
    results: list[tuple[np.ndarray, np.ndarray]] = []
    for half_window in half_windows:
        window = 2 * half_window + 1
        visible_count = len(visible_gr)
        hidden_count = len(hidden_gr)
        if visible_count < window + 1 or hidden_count == 0:
            fallback_tvt = np.full(hidden_count, visible_tvt[-1], dtype=np.float32)
            fallback_score = np.zeros(hidden_count, dtype=np.float32)
            results.append((fallback_tvt, fallback_score))
            continue

        smoothed_visible = (
            pd.Series(visible_gr)
            .rolling(5, center=True, min_periods=1)
            .mean()
            .to_numpy(dtype=np.float32)
        )
        smoothed_hidden = (
            pd.Series(hidden_gr)
            .rolling(5, center=True, min_periods=1)
            .mean()
            .to_numpy(dtype=np.float32)
        )
        starts = np.arange(0, visible_count - window + 1, stride, dtype=np.int32)
        if len(starts) == 0:
            fallback_tvt = np.full(hidden_count, visible_tvt[-1], dtype=np.float32)
            fallback_score = np.zeros(hidden_count, dtype=np.float32)
            results.append((fallback_tvt, fallback_score))
            continue

        candidate_windows = smoothed_visible[
            starts[:, None] + np.arange(window, dtype=np.int32)[None, :]
        ].astype(np.float32)
        normalized_candidates = (
            candidate_windows - candidate_windows.mean(axis=1, keepdims=True)
        ) / (candidate_windows.std(axis=1, keepdims=True) + 1e-6)
        padded_hidden = np.pad(smoothed_hidden, half_window, mode="edge")
        hidden_windows = padded_hidden[
            np.arange(hidden_count)[:, None] + np.arange(window)[None, :]
        ].astype(np.float32)
        normalized_hidden = (
            hidden_windows - hidden_windows.mean(axis=1, keepdims=True)
        ) / (hidden_windows.std(axis=1, keepdims=True) + 1e-6)
        correlations = normalized_hidden @ normalized_candidates.T / window
        best_indices = correlations.argmax(axis=1)
        best_scores = correlations.max(axis=1).astype(np.float32)
        best_tvt_indices = np.clip(
            starts[best_indices] + half_window,
            0,
            visible_count - 1,
        )
        results.append((visible_tvt[best_tvt_indices].astype(np.float32), best_scores))

    candidate_tvts = np.stack([result[0] for result in results], axis=1)
    candidate_scores = np.stack([result[1] for result in results], axis=1)
    score_weights = np.exp(3.0 * candidate_scores)
    score_weights /= score_weights.sum(axis=1, keepdims=True) + 1e-9
    ensemble = (candidate_tvts * score_weights).sum(axis=1).astype(np.float32)
    return results, ensemble


def _beam_delta_summaries(
    beam_paths: dict[str, np.ndarray],
    last_known_tvt: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """先转成相对最后可见 TVT 的路径，再按 notebook 顺序做汇总。"""

    anchor = np.float32(last_known_tvt)
    delta_matrix = np.stack(
        [(path - anchor).astype(np.float32) for path in beam_paths.values()],
        axis=1,
    )
    mean_delta = delta_matrix.mean(axis=1).astype(np.float32)
    std_delta = delta_matrix.std(axis=1).astype(np.float32)
    median_delta = np.median(delta_matrix, axis=1).astype(np.float32)
    return delta_matrix, mean_delta, std_delta, median_delta


def _prepare_inputs(
    horizontal_df: pd.DataFrame,
    typewell_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray]:
    required_horizontal = {"MD", "Z", "GR", "TVT_input"}
    required_typewell = {"TVT", "GR"}
    missing_horizontal = required_horizontal - set(horizontal_df.columns)
    missing_typewell = required_typewell - set(typewell_df.columns)
    if missing_horizontal:
        raise ValueError(f"水平井缺少列：{sorted(missing_horizontal)}")
    if missing_typewell:
        raise ValueError(f"Typewell 缺少列：{sorted(missing_typewell)}")

    horizontal = horizontal_df.copy()
    for column in required_horizontal:
        horizontal[column] = pd.to_numeric(horizontal[column], errors="coerce")
    visible_mask = horizontal["TVT_input"].notna().to_numpy()
    hidden_mask = ~visible_mask
    if int(visible_mask.sum()) < 10:
        raise ValueError("可见前缀少于 10 行，无法复现 notebook 候选")
    if not bool(hidden_mask.any()):
        raise ValueError("水平井没有 TVT_input 为空的隐藏行")
    first_hidden_position = int(np.flatnonzero(hidden_mask)[0])
    if visible_mask[first_hidden_position:].any():
        raise ValueError("TVT_input 必须是连续可见前缀，隐藏行必须位于井尾")
    if not np.isfinite(horizontal.loc[visible_mask, ["MD", "Z", "TVT_input"]]).all().all():
        raise ValueError("可见前缀的 MD、Z 或 TVT_input 含非有限值")
    if not np.isfinite(horizontal.loc[hidden_mask, ["MD", "Z"]]).all().all():
        raise ValueError("隐藏段的 MD 或 Z 含非有限值")

    typewell = typewell_df[["TVT", "GR"]].copy()
    typewell["TVT"] = pd.to_numeric(typewell["TVT"], errors="coerce")
    typewell["GR"] = pd.to_numeric(typewell["GR"], errors="coerce")
    typewell = typewell.dropna(subset=["TVT"]).sort_values("TVT").reset_index(drop=True)
    if len(typewell) < 3:
        raise ValueError("Typewell 有效 TVT 少于 3 行")
    typewell["GR"] = typewell["GR"].interpolate(limit_direction="both")
    if not np.isfinite(typewell[["TVT", "GR"]]).all().all():
        raise ValueError("Typewell 的 TVT 或 GR 无法得到有限值")
    return horizontal, typewell, visible_mask, hidden_mask


def build_candidate_features(
    horizontal_df: pd.DataFrame,
    typewell_df: pd.DataFrame,
    seed: int = 42,
) -> pd.DataFrame:
    """返回隐藏行的 ``row_index`` 与固定顺序的 24 个合法候选特征。"""

    horizontal, typewell, visible_mask, hidden_mask = _prepare_inputs(
        horizontal_df,
        typewell_df,
    )
    normalized_seed = int(seed)
    if normalized_seed < 0 or normalized_seed >= 2**31 - 1:
        raise ValueError("seed 必须位于 [0, 2**31-1) 内")

    visible = horizontal.loc[visible_mask]
    hidden = horizontal.loc[hidden_mask]
    typewell_tvt = typewell["TVT"].to_numpy(dtype=np.float64)
    typewell_gr = typewell["GR"].to_numpy(dtype=np.float64)
    last_known_tvt = float(visible["TVT_input"].iloc[-1])

    gr_sigma = _gr_sigma(horizontal, typewell_tvt, typewell_gr)
    visible_tail = visible.tail(30)
    tail_tvt_change = np.diff(visible_tail["TVT_input"].to_numpy(dtype=np.float64))
    tail_z_change = np.diff(visible_tail["Z"].to_numpy(dtype=np.float64))
    tail_md_change = np.diff(visible_tail["MD"].to_numpy(dtype=np.float64))
    valid_tail_steps = tail_md_change > 0
    if int(valid_tail_steps.sum()) >= 3:
        initial_structure_rate = float(
            np.median(
                (tail_tvt_change + tail_z_change)[valid_tail_steps]
                / tail_md_change[valid_tail_steps]
            )
        )
    else:
        initial_structure_rate = 0.0

    typewell_gr_grid, grid_minimum, grid_step = _regular_typewell_grid(
        typewell_tvt,
        typewell_gr,
    )
    hidden_md = hidden["MD"].to_numpy(dtype=np.float64)
    hidden_z = hidden["Z"].to_numpy(dtype=np.float64)
    hidden_raw_gr = hidden["GR"].to_numpy(dtype=np.float64)
    initial_structure_position = last_known_tvt + float(visible["Z"].iloc[-1])
    pf_ancc, pf_ancc_std = _particle_filter_ancc(
        hidden_md,
        hidden_z,
        hidden_raw_gr,
        typewell_gr_grid,
        grid_minimum,
        grid_step,
        gr_sigma,
        initial_structure_position,
        initial_structure_rate,
        ANCC_N,
        normalized_seed,
    )

    visible_z_change = np.diff(visible["Z"].to_numpy(dtype=np.float64))
    visible_tvt_change = np.diff(visible["TVT_input"].to_numpy(dtype=np.float64))
    visible_md_change = np.diff(visible["MD"].to_numpy(dtype=np.float64))
    valid_visible_steps = visible_md_change > 0
    if int(valid_visible_steps.sum()) >= 10:
        z_velocity = visible_z_change[valid_visible_steps] / visible_md_change[valid_visible_steps]
        tvt_velocity = (
            visible_tvt_change[valid_visible_steps] / visible_md_change[valid_visible_steps]
        )
        design = np.column_stack([z_velocity, np.ones_like(z_velocity)])
        coefficients = np.linalg.lstsq(design, tvt_velocity, rcond=None)[0]
        z_velocity_coefficient = float(coefficients[0])
        z_velocity_intercept = float(coefficients[1])
        z_velocity_residual = tvt_velocity - (
            z_velocity_coefficient * z_velocity + z_velocity_intercept
        )
        z_velocity_sigma = max(float(np.std(z_velocity_residual)), 0.001)
    else:
        z_velocity_coefficient = -1.0
        z_velocity_intercept = 0.0
        z_velocity_sigma = 0.1

    velocity_tail = visible.tail(20)
    velocity_tvt_change = np.diff(
        velocity_tail["TVT_input"].to_numpy(dtype=np.float64)
    )
    velocity_md_change = np.diff(velocity_tail["MD"].to_numpy(dtype=np.float64))
    valid_velocity_steps = velocity_md_change > 0
    if int(valid_velocity_steps.sum()) >= 3:
        initial_tvt_velocity = float(
            np.median(
                velocity_tvt_change[valid_velocity_steps]
                / velocity_md_change[valid_velocity_steps]
            )
        )
    else:
        initial_tvt_velocity = 0.0

    typewell_smoothed_gr = (
        pd.Series(typewell_gr)
        .rolling(PF_GR_WINDOW, center=True, min_periods=1)
        .mean()
        .to_numpy(dtype=np.float64)
    )
    typewell_smoothed_gr_grid, _, _ = _regular_typewell_grid(
        typewell_tvt,
        typewell_smoothed_gr,
    )
    horizontal_smoothed_gr = (
        horizontal["GR"]
        .rolling(PF_GR_WINDOW, center=True, min_periods=1)
        .mean()
        .to_numpy(dtype=np.float64)
    )
    hidden_smoothed_gr = horizontal_smoothed_gr[hidden_mask]
    pf_z, _pf_z_std = _particle_filter_z(
        hidden_md,
        hidden_z,
        hidden_raw_gr,
        hidden_smoothed_gr,
        typewell_gr_grid,
        typewell_smoothed_gr_grid,
        grid_minimum,
        grid_step,
        gr_sigma,
        last_known_tvt,
        initial_tvt_velocity,
        z_velocity_coefficient,
        z_velocity_intercept,
        z_velocity_sigma,
        PF_N,
        normalized_seed + 1,
    )

    gr_full = (
        horizontal["GR"]
        .interpolate(limit_direction="both")
        .fillna(float(np.nanmean(typewell_gr)))
        .to_numpy(dtype=np.float32)
    )
    hidden_gr = gr_full[hidden_mask]
    visible_gr = gr_full[visible_mask]
    visible_tvt = visible["TVT_input"].to_numpy(dtype=np.float32)

    beam_paths: dict[str, np.ndarray] = {}
    for beam_size, movement_cost, error_scale, smoothing_radius, tag in BEAM_CONFIGS:
        beam_paths[tag] = _beam_search(
            hidden_gr,
            typewell_tvt,
            typewell_gr,
            last_known_tvt,
            beam_size,
            movement_cost,
            error_scale,
            smoothing_radius,
        )
    (
        beam_delta_matrix,
        beam_mean_delta,
        beam_std_delta,
        beam_median_delta,
    ) = _beam_delta_summaries(beam_paths, last_known_tvt)
    beam_reference = (beam_paths["cons"] + beam_paths["sm5"]) / 2.0

    ncc_results, ncc_ensemble = _multi_scale_ncc(
        visible_gr,
        visible_tvt,
        hidden_gr,
    )
    sc8, sc8_score = ncc_results[0]
    sc15, sc15_score = ncc_results[1]
    sc25, sc25_score = ncc_results[2]
    ncc_consensus = (sc8 + sc15 + sc25) / 3.0
    ncc_trust = float(np.clip(len(visible) / 200.0, 0.0, 0.6))
    hybrid_reference = (
        (1.0 - ncc_trust) * beam_reference + ncc_trust * ncc_ensemble
    )

    hidden_count = len(hidden)
    candidate_data: dict[str, np.ndarray] = {
        "pf_ancc_delta": (pf_ancc - last_known_tvt).astype(np.float32),
        "pf_ancc_std": pf_ancc_std.astype(np.float32),
        "pf_z_delta": (pf_z - last_known_tvt).astype(np.float32),
        "pf_vs_z": (pf_ancc - pf_z).astype(np.float32),
        "beam_cons_d": (beam_paths["cons"] - last_known_tvt).astype(np.float32),
        "beam_loose_d": (beam_paths["loose"] - last_known_tvt).astype(np.float32),
        "beam_vcons_d": (beam_paths["vcons"] - last_known_tvt).astype(np.float32),
        "beam_sm5_d": (beam_paths["sm5"] - last_known_tvt).astype(np.float32),
        "beam_vloose_d": (beam_paths["vloose"] - last_known_tvt).astype(np.float32),
        "beam_mid_d": (beam_paths["mid"] - last_known_tvt).astype(np.float32),
        "beam_stiff_d": (beam_paths["stiff"] - last_known_tvt).astype(np.float32),
        "beam_mean_d": beam_mean_delta,
        "beam_std_d": beam_std_delta,
        "beam_med_d": beam_median_delta,
        "sc8_d": (sc8 - last_known_tvt).astype(np.float32),
        "sc8_sc": sc8_score.astype(np.float32),
        "sc15_d": (sc15 - last_known_tvt).astype(np.float32),
        "sc15_sc": sc15_score.astype(np.float32),
        "sc25_d": (sc25 - last_known_tvt).astype(np.float32),
        "sc25_sc": sc25_score.astype(np.float32),
        "sc_cons_d": (ncc_consensus - last_known_tvt).astype(np.float32),
        "sc_ens_d": (ncc_ensemble - last_known_tvt).astype(np.float32),
        "sc_trust": np.full(hidden_count, ncc_trust, dtype=np.float32),
        "hyb_d": (hybrid_reference - last_known_tvt).astype(np.float32),
    }
    result = pd.DataFrame(
        {"row_index": hidden.index.to_numpy(dtype=np.int64), **candidate_data}
    )
    values = result[DIRECT_CANDIDATE_COLUMNS].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all():
        raise ValueError("候选复现产生了 NaN 或 Inf")
    return result[["row_index", *DIRECT_CANDIDATE_COLUMNS]]


## 2. P01/P02 的 128-seed 粒子滤波路径

In [ ]:
"""P2-P01：复刻旧 Notebook 的多随机种子粒子滤波路径。

本模块只使用当前井在测试期可获得的信息：MD、Z、GR、TVT_input，
以及配对 Typewell 的 TVT/GR。隐藏 TVT 不会被读取。
"""

from __future__ import annotations

import math
from collections.abc import Mapping
from typing import Any

import numpy as np
import pandas as pd
from numba import njit


PF_FEATURE_COLUMNS = [
    "row_index",
    "last_visible_tvt",
    "pf128_mean_tvt",
    "pf128_mean_delta",
    "pf128_seed0_delta",
    "pf128_scale_3_delta",
    "pf128_scale_5_delta",
    "pf128_scale_8_delta",
    "pf128_scale_12_delta",
    "pf128_seed_std",
]


@njit(cache=False, nogil=True)
def _interpolate_regular_grid(
    grid: np.ndarray,
    value: float,
    grid_minimum: float,
    grid_step: float,
) -> float:
    """按 Notebook `_interp1` 的规则在线性等距网格上插值。"""

    left_index = int((value - grid_minimum) / grid_step)
    if left_index < 0:
        return grid[0]

    final_index = len(grid) - 1
    if left_index >= final_index:
        return grid[final_index]

    fractional_position = (value - grid_minimum) / grid_step - left_index
    return (
        grid[left_index] * (1.0 - fractional_position)
        + grid[left_index + 1] * fractional_position
    )


@njit(cache=False, nogil=True)
def particle_filter_all_seeds_numba(
    md: np.ndarray,
    z: np.ndarray,
    horizontal_gr: np.ndarray,
    typewell_gr_grid: np.ndarray,
    typewell_min_tvt: float,
    typewell_step_ft: float,
    gr_sigma: float,
    initial_u: float,
    initial_rate: float,
    number_of_particles: int,
    number_of_seeds: int,
    seed_base: int,
    rate_momentum: float,
    rate_noise: float,
    position_noise_ft: float,
    resample_position_noise_ft: float,
    resample_rate_noise: float,
    resample_effective_fraction: float,
    initial_position_spread_ft: float,
    position_limit_beyond_typewell_ft: float,
    initial_rate_std: float,
    minimum_md_step_ft: float,
    squared_gr_residual_cap: float,
    likelihood_floor: float,
) -> tuple[np.ndarray, np.ndarray]:
    """运行所有固定 seed；随机数调用顺序严格复刻 Notebook cell 37。

    返回：
    - `predictions`：形状 [seed 数, 隐藏行数]，每项是粒子加权 TVT；
    - `log_likelihoods`：形状 [seed 数]，每条完整路径的累计对数似然。
    """

    number_of_rows = len(md)
    predictions = np.empty((number_of_seeds, number_of_rows), dtype=np.float64)
    log_likelihoods = np.empty(number_of_seeds, dtype=np.float64)

    # 这里故意保留 Notebook 的定义：上限基准是 min + len(grid) * step。
    typewell_limit_maximum = (
        typewell_min_tvt + len(typewell_gr_grid) * typewell_step_ft
    )

    for seed_offset in range(number_of_seeds):
        np.random.seed(seed_base + seed_offset)

        particle_u_positions = np.empty(number_of_particles, dtype=np.float64)
        particle_rates = np.empty(number_of_particles, dtype=np.float64)
        particle_weights = (
            np.ones(number_of_particles, dtype=np.float64) / number_of_particles
        )

        # 必须逐粒子交替抽位置和倾角，不能向量化，否则随机序列会改变。
        for particle_index in range(number_of_particles):
            particle_u_positions[particle_index] = (
                initial_u + initial_position_spread_ft * np.random.randn()
            )
            particle_rates[particle_index] = (
                initial_rate + initial_rate_std * np.random.randn()
            )

        path_log_likelihood = 0.0
        previous_md = md[0] - minimum_md_step_ft

        for row_index in range(number_of_rows):
            md_change = md[row_index] - previous_md
            if md_change < minimum_md_step_ft:
                md_change = minimum_md_step_ft

            # 状态转移：先更新 U 沿 MD 的变化率，再把变化率积分到 U。
            for particle_index in range(number_of_particles):
                particle_rates[particle_index] = (
                    rate_momentum * particle_rates[particle_index]
                    + rate_noise * np.random.randn()
                )
                particle_u_positions[particle_index] += (
                    particle_rates[particle_index] * md_change
                    + position_noise_ft * np.random.randn()
                )

                particle_tvt = particle_u_positions[particle_index] - z[row_index]
                lower_limit = (
                    typewell_min_tvt - position_limit_beyond_typewell_ft
                )
                upper_limit = (
                    typewell_limit_maximum + position_limit_beyond_typewell_ft
                )
                if particle_tvt < lower_limit:
                    particle_tvt = lower_limit
                if particle_tvt > upper_limit:
                    particle_tvt = upper_limit
                particle_u_positions[particle_index] = particle_tvt + z[row_index]

            average_likelihood = 0.0
            for particle_index in range(number_of_particles):
                particle_tvt = particle_u_positions[particle_index] - z[row_index]
                expected_gr = _interpolate_regular_grid(
                    typewell_gr_grid,
                    particle_tvt,
                    typewell_min_tvt,
                    typewell_step_ft,
                )
                standardized_residual = (
                    horizontal_gr[row_index] - expected_gr
                ) / gr_sigma
                squared_residual = standardized_residual * standardized_residual
                if squared_residual > squared_gr_residual_cap:
                    squared_residual = squared_gr_residual_cap

                observation_likelihood = np.exp(-0.5 * squared_residual)
                if observation_likelihood < likelihood_floor:
                    observation_likelihood = likelihood_floor

                average_likelihood += (
                    particle_weights[particle_index] * observation_likelihood
                )
                particle_weights[particle_index] *= observation_likelihood

            if average_likelihood < likelihood_floor:
                average_likelihood = likelihood_floor
            path_log_likelihood += np.log(average_likelihood)

            weight_sum = 0.0
            for particle_index in range(number_of_particles):
                weight_sum += particle_weights[particle_index]
            if weight_sum > 0.0:
                for particle_index in range(number_of_particles):
                    particle_weights[particle_index] /= weight_sum
            else:
                for particle_index in range(number_of_particles):
                    particle_weights[particle_index] = 1.0 / number_of_particles

            inverse_effective_count = 0.0
            for particle_index in range(number_of_particles):
                inverse_effective_count += (
                    particle_weights[particle_index]
                    * particle_weights[particle_index]
                )
            effective_particle_count = 1.0 / inverse_effective_count

            if (
                effective_particle_count
                < resample_effective_fraction * number_of_particles
            ):
                cumulative_weights = np.empty(number_of_particles, dtype=np.float64)
                cumulative_weight = 0.0
                for particle_index in range(number_of_particles):
                    cumulative_weight += particle_weights[particle_index]
                    cumulative_weights[particle_index] = cumulative_weight

                first_systematic_position = np.random.uniform(
                    0.0, 1.0 / number_of_particles
                )
                new_positions = np.empty(number_of_particles, dtype=np.float64)
                new_rates = np.empty(number_of_particles, dtype=np.float64)
                source_index = 0

                for particle_index in range(number_of_particles):
                    systematic_position = (
                        first_systematic_position
                        + particle_index / number_of_particles
                    )
                    while (
                        source_index < number_of_particles - 1
                        and cumulative_weights[source_index] < systematic_position
                    ):
                        source_index += 1
                    new_positions[particle_index] = (
                        particle_u_positions[source_index]
                        + resample_position_noise_ft * np.random.randn()
                    )
                    new_rates[particle_index] = (
                        particle_rates[source_index]
                        + resample_rate_noise * np.random.randn()
                    )

                for particle_index in range(number_of_particles):
                    particle_u_positions[particle_index] = new_positions[particle_index]
                    particle_rates[particle_index] = new_rates[particle_index]
                    particle_weights[particle_index] = 1.0 / number_of_particles

            estimated_tvt = 0.0
            for particle_index in range(number_of_particles):
                estimated_tvt += particle_weights[particle_index] * (
                    particle_u_positions[particle_index] - z[row_index]
                )
            predictions[seed_offset, row_index] = estimated_tvt
            previous_md = md[row_index]

        log_likelihoods[seed_offset] = path_log_likelihood

    return predictions, log_likelihoods


def _interpolate_regular_grid_reference(
    grid: np.ndarray,
    value: float,
    grid_minimum: float,
    grid_step: float,
) -> float:
    """Python 参考版插值；保留向零截断的下标规则。"""

    left_index = int((value - grid_minimum) / grid_step)
    if left_index < 0:
        return float(grid[0])
    final_index = len(grid) - 1
    if left_index >= final_index:
        return float(grid[final_index])
    fractional_position = (value - grid_minimum) / grid_step - left_index
    return float(
        grid[left_index] * (1.0 - fractional_position)
        + grid[left_index + 1] * fractional_position
    )


def particle_filter_all_seeds_reference(
    md: np.ndarray,
    z: np.ndarray,
    horizontal_gr: np.ndarray,
    typewell_gr_grid: np.ndarray,
    typewell_min_tvt: float,
    typewell_step_ft: float,
    gr_sigma: float,
    initial_u: float,
    initial_rate: float,
    number_of_particles: int,
    number_of_seeds: int,
    seed_base: int,
    rate_momentum: float,
    rate_noise: float,
    position_noise_ft: float,
    resample_position_noise_ft: float,
    resample_rate_noise: float,
    resample_effective_fraction: float,
    initial_position_spread_ft: float,
    position_limit_beyond_typewell_ft: float,
    initial_rate_std: float,
    minimum_md_step_ft: float,
    squared_gr_residual_cap: float,
    likelihood_floor: float,
) -> tuple[np.ndarray, np.ndarray]:
    """便于单元测试的直接循环版；公式和随机数调用顺序与生产内核相同。"""

    number_of_rows = len(md)
    predictions = np.empty((number_of_seeds, number_of_rows), dtype=np.float64)
    log_likelihoods = np.empty(number_of_seeds, dtype=np.float64)
    typewell_limit_maximum = (
        typewell_min_tvt + len(typewell_gr_grid) * typewell_step_ft
    )

    for seed_offset in range(number_of_seeds):
        random_state = np.random.RandomState(seed_base + seed_offset)
        particle_u_positions = np.empty(number_of_particles, dtype=np.float64)
        particle_rates = np.empty(number_of_particles, dtype=np.float64)
        particle_weights = (
            np.ones(number_of_particles, dtype=np.float64) / number_of_particles
        )

        for particle_index in range(number_of_particles):
            particle_u_positions[particle_index] = (
                initial_u + initial_position_spread_ft * random_state.randn()
            )
            particle_rates[particle_index] = (
                initial_rate + initial_rate_std * random_state.randn()
            )

        path_log_likelihood = 0.0
        previous_md = md[0] - minimum_md_step_ft

        for row_index in range(number_of_rows):
            md_change = md[row_index] - previous_md
            if md_change < minimum_md_step_ft:
                md_change = minimum_md_step_ft

            for particle_index in range(number_of_particles):
                particle_rates[particle_index] = (
                    rate_momentum * particle_rates[particle_index]
                    + rate_noise * random_state.randn()
                )
                particle_u_positions[particle_index] += (
                    particle_rates[particle_index] * md_change
                    + position_noise_ft * random_state.randn()
                )

                particle_tvt = particle_u_positions[particle_index] - z[row_index]
                lower_limit = (
                    typewell_min_tvt - position_limit_beyond_typewell_ft
                )
                upper_limit = (
                    typewell_limit_maximum + position_limit_beyond_typewell_ft
                )
                if particle_tvt < lower_limit:
                    particle_tvt = lower_limit
                if particle_tvt > upper_limit:
                    particle_tvt = upper_limit
                particle_u_positions[particle_index] = particle_tvt + z[row_index]

            average_likelihood = 0.0
            for particle_index in range(number_of_particles):
                particle_tvt = particle_u_positions[particle_index] - z[row_index]
                expected_gr = _interpolate_regular_grid_reference(
                    typewell_gr_grid,
                    particle_tvt,
                    typewell_min_tvt,
                    typewell_step_ft,
                )
                standardized_residual = (
                    horizontal_gr[row_index] - expected_gr
                ) / gr_sigma
                squared_residual = standardized_residual * standardized_residual
                if squared_residual > squared_gr_residual_cap:
                    squared_residual = squared_gr_residual_cap
                observation_likelihood = math.exp(-0.5 * squared_residual)
                if observation_likelihood < likelihood_floor:
                    observation_likelihood = likelihood_floor
                average_likelihood += (
                    particle_weights[particle_index] * observation_likelihood
                )
                particle_weights[particle_index] *= observation_likelihood

            if average_likelihood < likelihood_floor:
                average_likelihood = likelihood_floor
            path_log_likelihood += math.log(average_likelihood)

            weight_sum = 0.0
            for particle_index in range(number_of_particles):
                weight_sum += particle_weights[particle_index]
            if weight_sum > 0.0:
                for particle_index in range(number_of_particles):
                    particle_weights[particle_index] /= weight_sum
            else:
                for particle_index in range(number_of_particles):
                    particle_weights[particle_index] = 1.0 / number_of_particles

            inverse_effective_count = 0.0
            for particle_index in range(number_of_particles):
                inverse_effective_count += (
                    particle_weights[particle_index]
                    * particle_weights[particle_index]
                )
            effective_particle_count = 1.0 / inverse_effective_count

            if (
                effective_particle_count
                < resample_effective_fraction * number_of_particles
            ):
                cumulative_weights = np.empty(number_of_particles, dtype=np.float64)
                cumulative_weight = 0.0
                for particle_index in range(number_of_particles):
                    cumulative_weight += particle_weights[particle_index]
                    cumulative_weights[particle_index] = cumulative_weight
                first_systematic_position = random_state.uniform(
                    0.0, 1.0 / number_of_particles
                )
                new_positions = np.empty(number_of_particles, dtype=np.float64)
                new_rates = np.empty(number_of_particles, dtype=np.float64)
                source_index = 0
                for particle_index in range(number_of_particles):
                    systematic_position = (
                        first_systematic_position
                        + particle_index / number_of_particles
                    )
                    while (
                        source_index < number_of_particles - 1
                        and cumulative_weights[source_index] < systematic_position
                    ):
                        source_index += 1
                    new_positions[particle_index] = (
                        particle_u_positions[source_index]
                        + resample_position_noise_ft * random_state.randn()
                    )
                    new_rates[particle_index] = (
                        particle_rates[source_index]
                        + resample_rate_noise * random_state.randn()
                    )
                for particle_index in range(number_of_particles):
                    particle_u_positions[particle_index] = new_positions[particle_index]
                    particle_rates[particle_index] = new_rates[particle_index]
                    particle_weights[particle_index] = 1.0 / number_of_particles

            estimated_tvt = 0.0
            for particle_index in range(number_of_particles):
                estimated_tvt += particle_weights[particle_index] * (
                    particle_u_positions[particle_index] - z[row_index]
                )
            predictions[seed_offset, row_index] = estimated_tvt
            previous_md = md[row_index]

        log_likelihoods[seed_offset] = path_log_likelihood

    return predictions, log_likelihoods


def _require_columns(table: pd.DataFrame, columns: list[str], table_name: str) -> None:
    """在进入数值代码前给出清楚的缺列错误。"""

    missing_columns = [column for column in columns if column not in table.columns]
    if missing_columns:
        raise ValueError(f"{table_name} 缺少列: {missing_columns}")


def _finite_float_array(values: Any, name: str) -> np.ndarray:
    """统一转为连续 float64，并阻止 NaN/无穷进入 PF 内核。"""

    array = np.ascontiguousarray(np.asarray(values, dtype=np.float64))
    if not np.isfinite(array).all():
        raise ValueError(f"{name} 含 NaN 或无穷值")
    return array


def prepare_particle_filter_inputs(
    horizontal_well: pd.DataFrame,
    typewell: pd.DataFrame,
    parameters: Mapping[str, Any],
) -> dict[str, Any]:
    """把一口井整理成 PF 输入；返回字典刻意不含隐藏 `TVT`。"""

    _require_columns(horizontal_well, ["MD", "Z", "GR", "TVT_input"], "水平井")
    _require_columns(typewell, ["TVT", "GR"], "Typewell")

    visible_mask = horizontal_well["TVT_input"].notna().to_numpy()
    hidden_mask = ~visible_mask
    visible_positions = np.flatnonzero(visible_mask)
    hidden_positions = np.flatnonzero(hidden_mask)
    if len(visible_positions) == 0:
        raise ValueError("水平井没有可见 TVT_input，无法初始化粒子")
    if len(hidden_positions) == 0:
        raise ValueError("水平井没有自然隐藏行，无法生成路径特征")

    # 比赛数据的可见段必须是前缀；若中间有洞，Notebook 的 last 语义会含糊。
    if visible_positions[-1] >= hidden_positions[0]:
        raise ValueError("TVT_input 可见行不是连续前缀")

    sorted_typewell = typewell.sort_values("TVT", kind="mergesort")
    typewell_tvt = sorted_typewell["TVT"].to_numpy(dtype=np.float64, copy=True)
    typewell_gr_series = sorted_typewell["GR"].astype(np.float64)
    typewell_gr_mean = float(typewell_gr_series.mean(skipna=True))
    if not np.isfinite(typewell_gr_mean):
        raise ValueError("Typewell GR 全部缺失")
    typewell_gr = typewell_gr_series.fillna(typewell_gr_mean).to_numpy(copy=True)
    typewell_tvt = _finite_float_array(typewell_tvt, "Typewell TVT")
    typewell_gr = _finite_float_array(typewell_gr, "Typewell GR")
    if len(typewell_tvt) < 2:
        raise ValueError("Typewell 至少需要两个 TVT/GR 点")
    if np.any(np.diff(typewell_tvt) <= 0.0):
        raise ValueError("Typewell TVT 必须严格递增且不能重复")

    grid_step = float(parameters["typewell_grid_step_ft"])
    if not np.isfinite(grid_step) or grid_step <= 0.0:
        raise ValueError("typewell_grid_step_ft 必须为正数")
    typewell_min_tvt = float(typewell_tvt[0])
    typewell_grid_tvt = np.arange(
        typewell_min_tvt,
        float(typewell_tvt[-1]) + grid_step,
        grid_step,
        dtype=np.float64,
    )
    typewell_gr_grid = _finite_float_array(
        np.interp(typewell_grid_tvt, typewell_tvt, typewell_gr),
        "Typewell 等距 GR 网格",
    )

    visible_tvt = _finite_float_array(
        horizontal_well.loc[visible_mask, "TVT_input"].to_numpy(),
        "可见 TVT_input",
    )
    visible_z = _finite_float_array(
        horizontal_well.loc[visible_mask, "Z"].to_numpy(),
        "可见 Z",
    )
    visible_md = _finite_float_array(
        horizontal_well.loc[visible_mask, "MD"].to_numpy(),
        "可见 MD",
    )

    # 严格复刻 lik_pf：可见 GR 缺失填 0，再与 Typewell GR 算总体标准差。
    visible_gr = (
        horizontal_well.loc[visible_mask, "GR"]
        .astype(np.float64)
        .fillna(0.0)
        .to_numpy()
    )
    typewell_gr_at_visible_tvt = np.interp(
        visible_tvt, typewell_tvt, typewell_gr
    )
    residual_sigma = float(
        np.nanstd(visible_gr - typewell_gr_at_visible_tvt)
    )
    gr_sigma = float(
        np.clip(
            residual_sigma,
            float(parameters["gr_sigma_min_api"]),
            float(parameters["gr_sigma_max_api"]),
        )
    )
    if not np.isfinite(gr_sigma) or gr_sigma <= 0.0:
        raise ValueError("由可见前缀得到的 GR sigma 非法")

    tail_rows = int(parameters["initial_rate_visible_tail_rows"])
    tail_start = max(0, len(visible_tvt) - tail_rows)
    tail_tvt = visible_tvt[tail_start:]
    tail_z = visible_z[tail_start:]
    tail_md = visible_md[tail_start:]
    tvt_change = np.diff(tail_tvt)
    z_change = np.diff(tail_z)
    md_change = np.diff(tail_md)
    valid_rate_change = md_change > 0.0
    if int(valid_rate_change.sum()) >= 3:
        initial_rate = float(
            np.median(
                (tvt_change[valid_rate_change] + z_change[valid_rate_change])
                / md_change[valid_rate_change]
            )
        )
    else:
        initial_rate = 0.0

    last_visible_position = int(visible_positions[-1])
    last_visible_tvt = float(visible_tvt[-1])
    initial_u = last_visible_tvt + float(visible_z[-1])

    # 和 Notebook 一样，先在整井行序上双向线性插值水平井 GR。
    interpolated_horizontal_gr = (
        horizontal_well["GR"]
        .astype(np.float64)
        .interpolate(limit_direction="both")
        .fillna(typewell_gr_mean)
        .to_numpy()
    )

    prepared: dict[str, Any] = {
        "md": _finite_float_array(
            horizontal_well.loc[hidden_mask, "MD"].to_numpy(), "隐藏段 MD"
        ),
        "z": _finite_float_array(
            horizontal_well.loc[hidden_mask, "Z"].to_numpy(), "隐藏段 Z"
        ),
        "horizontal_gr": _finite_float_array(
            interpolated_horizontal_gr[hidden_positions], "隐藏段插值 GR"
        ),
        "typewell_gr_grid": typewell_gr_grid,
        "typewell_min_tvt": typewell_min_tvt,
        "typewell_step_ft": grid_step,
        "gr_sigma": gr_sigma,
        "initial_u": initial_u,
        "initial_rate": initial_rate,
        "number_of_particles": int(parameters["number_of_particles"]),
        "number_of_seeds": int(parameters["number_of_seeds"]),
        "seed_base": int(parameters["seed_base"]),
        "rate_momentum": float(parameters["rate_momentum"]),
        "rate_noise": float(parameters["rate_noise"]),
        "position_noise_ft": float(parameters["position_noise_ft"]),
        "resample_position_noise_ft": float(
            parameters["resample_position_noise_ft"]
        ),
        "resample_rate_noise": float(parameters["resample_rate_noise"]),
        "resample_effective_fraction": float(
            parameters["resample_effective_fraction"]
        ),
        "initial_position_spread_ft": float(
            parameters["initial_position_spread_ft"]
        ),
        "position_limit_beyond_typewell_ft": float(
            parameters["position_limit_beyond_typewell_ft"]
        ),
        "initial_rate_std": float(parameters["initial_rate_std"]),
        "minimum_md_step_ft": float(parameters["minimum_md_step_ft"]),
        "squared_gr_residual_cap": float(
            parameters["squared_gr_residual_cap"]
        ),
        "likelihood_floor": float(parameters["likelihood_floor"]),
        "row_index": hidden_positions.astype(np.int64, copy=False),
        "last_visible_tvt": last_visible_tvt,
        "last_visible_position": last_visible_position,
    }
    return prepared


def _kernel_arguments(prepared: Mapping[str, Any]) -> dict[str, Any]:
    """从准备结果中只取生产内核声明的参数。"""

    return {
        "md": prepared["md"],
        "z": prepared["z"],
        "horizontal_gr": prepared["horizontal_gr"],
        "typewell_gr_grid": prepared["typewell_gr_grid"],
        "typewell_min_tvt": prepared["typewell_min_tvt"],
        "typewell_step_ft": prepared["typewell_step_ft"],
        "gr_sigma": prepared["gr_sigma"],
        "initial_u": prepared["initial_u"],
        "initial_rate": prepared["initial_rate"],
        "number_of_particles": prepared["number_of_particles"],
        "number_of_seeds": prepared["number_of_seeds"],
        "seed_base": prepared["seed_base"],
        "rate_momentum": prepared["rate_momentum"],
        "rate_noise": prepared["rate_noise"],
        "position_noise_ft": prepared["position_noise_ft"],
        "resample_position_noise_ft": prepared["resample_position_noise_ft"],
        "resample_rate_noise": prepared["resample_rate_noise"],
        "resample_effective_fraction": prepared["resample_effective_fraction"],
        "initial_position_spread_ft": prepared["initial_position_spread_ft"],
        "position_limit_beyond_typewell_ft": prepared[
            "position_limit_beyond_typewell_ft"
        ],
        "initial_rate_std": prepared["initial_rate_std"],
        "minimum_md_step_ft": prepared["minimum_md_step_ft"],
        "squared_gr_residual_cap": prepared["squared_gr_residual_cap"],
        "likelihood_floor": prepared["likelihood_floor"],
    }


def _likelihood_weighted_path(
    seed_predictions: np.ndarray,
    seed_log_likelihoods: np.ndarray,
    scale: float,
) -> np.ndarray:
    """按 Notebook 的温度缩放公式，将多 seed 路径变成一条加权路径。"""

    centered_log_likelihoods = seed_log_likelihoods - np.max(
        seed_log_likelihoods
    )
    seed_weights = np.exp(centered_log_likelihoods / float(scale))
    seed_weights /= seed_weights.sum()
    return (seed_weights[:, None] * seed_predictions).sum(axis=0)


def build_multiseed_pf_features(
    horizontal_well: pd.DataFrame,
    typewell: pd.DataFrame,
    parameters: Mapping[str, Any],
) -> tuple[pd.DataFrame, dict[str, float]]:
    """生成正式均值特征、四条审计 scale 路径和三个井级质量量。"""

    prepared = prepare_particle_filter_inputs(horizontal_well, typewell, parameters)
    seed_predictions, seed_log_likelihoods = particle_filter_all_seeds_numba(
        **_kernel_arguments(prepared)
    )

    # 旧特征表先把绝对路径压成 float32，再用 float32 的末值计算差值。
    mean_tvt = seed_predictions.mean(axis=0).astype(np.float32)
    seed0_tvt = seed_predictions[0].astype(np.float32)
    seed_std = seed_predictions.std(axis=0).astype(np.float32)
    last_visible_tvt = np.float32(prepared["last_visible_tvt"])
    number_of_hidden_rows = len(prepared["row_index"])

    scale_paths: dict[float, np.ndarray] = {}
    for scale_value in parameters["likelihood_scales"]:
        numeric_scale = float(scale_value)
        scale_paths[numeric_scale] = _likelihood_weighted_path(
            seed_predictions, seed_log_likelihoods, numeric_scale
        ).astype(np.float32)

    required_scales = (3.0, 5.0, 8.0, 12.0)
    missing_scales = [scale for scale in required_scales if scale not in scale_paths]
    if missing_scales:
        raise ValueError(f"likelihood_scales 缺少冻结值: {missing_scales}")

    features = pd.DataFrame(
        {
            "row_index": prepared["row_index"],
            "last_visible_tvt": np.full(
                number_of_hidden_rows, last_visible_tvt, dtype=np.float32
            ),
            "pf128_mean_tvt": mean_tvt,
            "pf128_mean_delta": mean_tvt - last_visible_tvt,
            "pf128_seed0_delta": seed0_tvt - last_visible_tvt,
            "pf128_scale_3_delta": scale_paths[3.0] - last_visible_tvt,
            "pf128_scale_5_delta": scale_paths[5.0] - last_visible_tvt,
            "pf128_scale_8_delta": scale_paths[8.0] - last_visible_tvt,
            "pf128_scale_12_delta": scale_paths[12.0] - last_visible_tvt,
            "pf128_seed_std": seed_std,
        },
        columns=PF_FEATURE_COLUMNS,
    )

    if not np.isfinite(features.drop(columns="row_index").to_numpy()).all():
        raise RuntimeError("PF 生成了 NaN 或无穷特征")

    quality = {
        "pf_best_ll_per_row": float(np.max(seed_log_likelihoods))
        / number_of_hidden_rows,
        "pf_ll_spread": float(np.std(seed_log_likelihoods)),
        "pf_gr_sigma": float(prepared["gr_sigma"]),
    }
    if not np.isfinite(np.asarray(list(quality.values()), dtype=np.float64)).all():
        raise RuntimeError("PF 生成了 NaN 或无穷质量指标")

    return features, quality


## 3. 测试特征组装

In [ ]:

def find_competition_root() -> Path:
    """寻找 Kaggle 比赛数据目录；本地运行时回退到项目 input/data/raw。"""
    fixed_candidates = [
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
        Path.cwd() / "input" / "data" / "raw",
    ]
    for candidate in fixed_candidates:
        if (candidate / "test").is_dir() and (candidate / "sample_submission.csv").is_file():
            return candidate
    for sample_path_text in glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True):
        candidate = Path(sample_path_text).parent
        if (candidate / "test").is_dir():
            return candidate
    raise FileNotFoundError("没有找到带 test/ 和 sample_submission.csv 的比赛数据目录")


def build_base_test_features(horizontal: pd.DataFrame) -> pd.DataFrame:
    """只用测试时可见列构造 B00 的 12 个基础特征，不读取隐藏 TVT。"""
    required = {"MD", "X", "Y", "Z", "GR", "TVT_input"}
    missing = required.difference(horizontal.columns)
    if missing:
        raise ValueError(f"水平井缺列：{sorted(missing)}")

    visible_mask = horizontal["TVT_input"].notna().to_numpy()
    hidden_mask = ~visible_mask
    visible_positions = np.flatnonzero(visible_mask)
    hidden_positions = np.flatnonzero(hidden_mask)
    if len(visible_positions) == 0 or len(hidden_positions) == 0:
        raise ValueError("水平井必须同时包含可见前缀和隐藏后缀")
    last_visible_position = int(visible_positions[-1])
    if not np.array_equal(visible_positions, np.arange(last_visible_position + 1)):
        raise ValueError("TVT_input 可见区不是连续前缀")
    if not np.array_equal(hidden_positions, np.arange(last_visible_position + 1, len(horizontal))):
        raise ValueError("TVT_input 隐藏区不是连续后缀")

    md_all = horizontal["MD"].to_numpy(dtype=np.float64)
    if np.any(np.diff(md_all) < 0.0):
        raise ValueError("MD 不是单调非降")

    hidden = horizontal.iloc[hidden_positions]
    last_visible = horizontal.iloc[last_visible_position]
    hidden_md = hidden["MD"].to_numpy(dtype=np.float64)
    hidden_span = max(float(hidden_md[-1] - hidden_md[0]), 1.0)
    dx = hidden["X"].to_numpy(dtype=np.float64) - float(last_visible["X"])
    dy = hidden["Y"].to_numpy(dtype=np.float64) - float(last_visible["Y"])
    dz = hidden["Z"].to_numpy(dtype=np.float64) - float(last_visible["Z"])
    gr_raw = hidden["GR"].to_numpy(dtype=np.float64)
    last_visible_tvt = float(last_visible["TVT_input"])

    return pd.DataFrame(
        {
            "row_index": hidden_positions.astype(np.int64),
            "last_visible_tvt": last_visible_tvt,
            "md_since_visible_end": hidden_md - float(last_visible["MD"]),
            "hidden_fraction": (hidden_md - hidden_md[0]) / hidden_span,
            "x_current": hidden["X"].to_numpy(dtype=np.float64),
            "y_current": hidden["Y"].to_numpy(dtype=np.float64),
            "z_current": hidden["Z"].to_numpy(dtype=np.float64),
            "dx_from_visible_end": dx,
            "dy_from_visible_end": dy,
            "dz_from_visible_end": dz,
            "dxy_from_visible_end": np.sqrt(dx * dx + dy * dy),
            "gr_raw": gr_raw,
            "gr_missing": np.isnan(gr_raw).astype(np.float64),
        }
    )


def build_one_test_well_features(
    well_id: str,
    horizontal_path: Path,
    typewell_path: Path,
) -> pd.DataFrame:
    """为一口测试井生成 12+24+5=41 个正式特征。"""
    horizontal = pd.read_csv(horizontal_path)
    typewell = pd.read_csv(typewell_path)

    base = build_base_test_features(horizontal)
    candidates = build_candidate_features(
        horizontal[["MD", "Z", "GR", "TVT_input"]].copy(),
        typewell[["TVT", "GR"]].copy(),
        seed=42,
    )
    pf_features, _quality = build_multiseed_pf_features(
        horizontal[["MD", "Z", "GR", "TVT_input"]].copy(),
        typewell[["TVT", "GR"]].copy(),
        PF_PARAMETERS,
    )
    selected_pf = pf_features[
        [
            "row_index",
            "pf128_mean_delta",
            "pf128_scale_3_delta",
            "pf128_scale_5_delta",
            "pf128_scale_8_delta",
            "pf128_scale_12_delta",
        ]
    ]

    merged = base.merge(candidates, on="row_index", how="left", validate="one_to_one")
    merged = merged.merge(selected_pf, on="row_index", how="left", validate="one_to_one")
    merged.insert(0, "well_id", str(well_id))
    merged.insert(1, "id", str(well_id) + "_" + merged["row_index"].astype(str))
    illegal_columns = [column for column in FEATURE_COLUMNS if column not in merged.columns]
    if illegal_columns:
        raise ValueError(f"{well_id} 缺少正式特征：{illegal_columns}")
    feature_values = merged[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    for column_index, column_name in enumerate(FEATURE_COLUMNS):
        values = feature_values[:, column_index]
        if column_name == "gr_raw":
            if np.isinf(values).any():
                raise ValueError(f"{well_id} 的 gr_raw 含 Inf")
        elif not np.isfinite(values).all():
            raise ValueError(f"{well_id} 的 {column_name} 含 NaN 或 Inf")
    return merged


## 4. ? Kaggle Dataset ???? P2-P02 fold ??

??? `rogii-p2-p02-models.zip` ?? Kaggle Dataset???? **Add Input** ???

In [ ]:
import zipfile


def find_model_source() -> tuple[str, Path]:
    """?? Dataset ?? ZIP??? Kaggle ???? ZIP ???????"""
    local_zip = Path.cwd() / "rogii-p2-p02-models.zip"
    if local_zip.is_file():
        return "zip", local_zip
    input_root = Path("/kaggle/input")
    zip_candidates = sorted(input_root.glob("**/rogii-p2-p02-models.zip"))
    if not zip_candidates:
        zip_candidates = sorted(input_root.glob("**/*p02*models*.zip"))
    if zip_candidates:
        return "zip", zip_candidates[0]
    direct_candidates = [Path.cwd() / "kaggle_assets" / "rogii-p2-p02-models"]
    direct_candidates += [path.parents[1] for path in input_root.glob("**/fold_0/model.txt")]
    for candidate in direct_candidates:
        if all((candidate / f"fold_{fold_id}" / "model.txt").is_file() for fold_id in range(5)):
            return "directory", candidate
    raise FileNotFoundError(
        "???? P2-P02 ????? rogii-p2-p02-models.zip ?? Kaggle Dataset?"
        "???? Notebook ? Add Input ???"
    )


source_kind, model_source = find_model_source()
MODELS = []
if source_kind == "zip":
    with zipfile.ZipFile(model_source, "r") as model_zip:
        archive_names = set(model_zip.namelist())
        for fold_id in range(5):
            expected_name = f"fold_{fold_id}/model.txt"
            matches = [
                name for name in archive_names
                if name.replace("\\", "/").endswith(expected_name)
            ]
            if len(matches) != 1:
                raise FileNotFoundError(f"?? ZIP ?? {expected_name}")
            model_text = model_zip.read(matches[0]).decode("utf-8")
            MODELS.append(lgb.Booster(model_str=model_text))
            print(f"??? fold {fold_id}????{MODELS[-1].num_trees()}")
else:
    for fold_id in range(5):
        model_path = model_source / f"fold_{fold_id}" / "model.txt"
        MODELS.append(lgb.Booster(model_file=str(model_path)))
        print(f"??? fold {fold_id}????{MODELS[-1].num_trees()}")

print("?????", model_source)


## 5. 生成 submission.csv

In [ ]:

start_time = time.time()
data_root = find_competition_root()
test_dir = data_root / "test"
sample = pd.read_csv(data_root / "sample_submission.csv")[["id"]].copy()
sample["id"] = sample["id"].astype(str)

horizontal_paths = sorted(test_dir.glob("*__horizontal_well.csv"))
if not horizontal_paths:
    raise FileNotFoundError(f"{test_dir} 中没有水平井 CSV")

prediction_parts = []
for well_number, horizontal_path in enumerate(horizontal_paths, start=1):
    well_id = horizontal_path.stem.replace("__horizontal_well", "")
    typewell_path = test_dir / f"{well_id}__typewell.csv"
    if not typewell_path.is_file():
        raise FileNotFoundError(typewell_path)
    print(f"[{well_number}/{len(horizontal_paths)}] 生成 {well_id} 的 41 个特征", flush=True)
    feature_table = build_one_test_well_features(well_id, horizontal_path, typewell_path)
    feature_matrix = feature_table[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    fold_predictions = np.column_stack(
        [model.predict(feature_matrix) for model in MODELS]
    )
    predicted_delta = fold_predictions.mean(axis=1)
    predicted_tvt = (
        feature_table["last_visible_tvt"].to_numpy(dtype=np.float64)
        + predicted_delta
    )
    prediction_parts.append(
        pd.DataFrame({"id": feature_table["id"].astype(str), "tvt": predicted_tvt})
    )

all_predictions = pd.concat(prediction_parts, ignore_index=True)
if all_predictions["id"].duplicated().any():
    raise RuntimeError("测试预测含重复 id")
submission = sample.merge(all_predictions, on="id", how="left", validate="one_to_one")
if submission["tvt"].isna().any() or not np.isfinite(submission["tvt"].to_numpy()).all():
    missing_examples = submission.loc[submission["tvt"].isna(), "id"].head().tolist()
    raise RuntimeError(f"submission 存在缺失预测：{missing_examples}")

output_path = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").is_dir() else Path("submission.csv")
submission[["id", "tvt"]].to_csv(output_path, index=False)
print(f"完成：{output_path}，{len(submission):,} 行，总耗时 {time.time() - start_time:.1f} 秒")
submission.head()
